In [ ]:
# ============================================
# AML DATA PREPROCESSING
# ============================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------

df = pd.read_csv("/content/aml_data/SAML-D.csv")

print("Original dataset shape:", df.shape)


# ------------------------------------------------
# 2. Handle Duplicate Transactions
# ------------------------------------------------

print("Duplicate rows before removal:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after removal:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)


# ------------------------------------------------
# 3. Check Missing Values
# ------------------------------------------------

missing_values = df.isnull().sum()

print("\nMissing values:")
print(missing_values)

print("\nTotal missing values:", df.isnull().sum().sum())


# ------------------------------------------------
# 4. Detect Invalid Transaction Amounts
# ------------------------------------------------

if "Amount" in df.columns:
    print("\nNegative amount transactions:",
          (df["Amount"] < 0).sum())

    print("Zero amount transactions:",
          (df["Amount"] == 0).sum())


# ------------------------------------------------
# 5. Detect Outliers Using IQR
# ------------------------------------------------

if "Amount" in df.columns:

    Q1 = df["Amount"].quantile(0.25)
    Q3 = df["Amount"].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df["Amount"] < lower_bound) |
        (df["Amount"] > upper_bound)
    ]

    print("\nNumber of Amount outliers:", len(outliers))
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)


# ------------------------------------------------
# 6. Define Features and Target
# ------------------------------------------------

X = df.drop("Is_laundering", axis=1)
y = df["Is_laundering"]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ------------------------------------------------
# 7. Identify Numerical and Categorical Features
# ------------------------------------------------

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


# ------------------------------------------------
# 8. Train-Test Split
# ------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)


# ------------------------------------------------
# 9. Encode Categorical Variables
#    and Scale Numerical Variables
# ------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ]
)


# ------------------------------------------------
# 10. Fit Preprocessor on Training Data
# ------------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("\nProcessed training shape:",
      X_train_processed.shape)

print("Processed testing shape:",
      X_test_processed.shape)


# ------------------------------------------------
# 11. Summary
# ------------------------------------------------

print("\n========== PREPROCESSING SUMMARY ==========")
print("Duplicates removed")
print("Missing values checked")
print("Amount outliers detected using IQR")
print("Categorical variables encoded using OneHotEncoder")
print("Numerical variables scaled using StandardScaler")
print("Train-test split completed")
print("===========================================")